# Intro to LLMs — Prompts, Roles, and Conversations

This notebook isn't about stocks. It's about the actual mechanics underneath
every LLM feature you'll build from here on — the news sentiment panel, the
dashboard chat agent, everything. Get comfortable here first.

**One package for all three providers.** OpenAI, Anthropic, and Gemini each
publish an "OpenAI-compatible" endpoint, so the `openai` Python package can
talk to any of them — you only ever change the API key, the `base_url`, and
the model name. No `anthropic` package, no `google.generativeai` package
(which is deprecated anyway — if you've seen that warning, this sidesteps it
completely).

**Three things you'll actually understand by the end:**
1. What a "role" is, and why `system` / `user` / `assistant` exist as separate things.
2. That an LLM has **no memory** — "conversation" is an illusion you build yourself.
3. That a system prompt is **optional** — a lever you choose to pull, not something every call needs.


## Setup
Free key: https://aistudio.google.com/apikey — then set `GEMINI_API_KEY`.

## 1. Setup — `.env` and picking a provider

```
OPENAI_API_KEY=sk-...
ANTHROPIC_API_KEY=sk-ant-...
GEMINI_API_KEY=...
```

**Honest note on cost:** OpenAI and Anthropic both require a payment method on
file for real usage — neither has a lasting free tier the way Gemini does
(1,500 free requests/day, no card required). If you don't have credits, use
Gemini, or set `PROVIDER = "mock"` and this notebook still runs end to end —
every real call gets replaced with a clearly-labeled fake response.

In [5]:
%pip install openai

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   -------------------------------


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
import os
from dotenv import load_dotenv
load_dotenv()

# ---- choose your provider here ----
PROVIDER = "gemini"   # one of: "openai", "anthropic", "gemini", "mock"

CONFIG = {
    "openai":    {"api_key": os.environ.get("OPENAI_API_KEY"),    "base_url": None,
                  "model": "gpt-5-mini"},
    "anthropic": {"api_key": os.environ.get("ANTHROPIC_API_KEY"), "base_url": "https://api.anthropic.com/v1/",
                  "model": "claude-sonnet-5"},
    "gemini":    {"api_key": os.environ.get("GEMINI_API_KEY"),    "base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
                  "model": "gemini-flash-latest"},   # alias -- always points at Google's current recommended flash model
}

print(f'provider: {PROVIDER}')

provider: gemini


## 2. The three roles

Every LLM chat API organizes a conversation as a list of messages, and every
message has a **role**:

- **`system`** — instructions about *how the model should behave*, set once.
  **This one is optional** — leave it out, and the model just uses its own
  default behavior. You'll see both cases below, back to back.
- **`user`** — what the person actually typed.
- **`assistant`** — what the model said back. You also use this role
  yourself when showing the model its own prior reply (section 5).

In [23]:
def chat(messages, system=None, provider=PROVIDER, max_tokens=1000, verbose=False):
    """
    Send a list of {"role":..., "content":...} messages, get back the reply text.
    `system` is genuinely optional -- pass nothing, and no system message is sent at all.

    verbose=True prints WHY the response ended -- "stop" (finished naturally)
    vs "length" (cut off by max_tokens). Use this instead of guessing whether
    a short or odd-looking answer is a truncation problem or something else.
    """
    if provider == "mock":
        last_user = next((m['content'] for m in reversed(messages) if m['role']=='user'), '')
        sys_note = f' [as instructed: {system[:40]}...]' if system else ' [no system prompt given]'
        return f"[MOCK REPLY]{sys_note} You said: \"{last_user[:60]}\" -- imagine a real, helpful answer here."

    from openai import OpenAI
    cfg = CONFIG[provider]
    client = OpenAI(api_key=cfg["api_key"], base_url=cfg["base_url"]) if cfg["base_url"] else OpenAI(api_key=cfg["api_key"])

    full_messages = ([{"role": "system", "content": system}] if system else []) + messages
    resp = client.chat.completions.create(model=cfg["model"], messages=full_messages, max_tokens=max_tokens)

    choice = resp.choices[0]
    if verbose:
        print(f"[finish_reason: {choice.finish_reason}]")
        if choice.finish_reason == "length":
            print(f"[TRUNCATED -- hit the {max_tokens}-token limit before finishing. Raise max_tokens.]")
    return choice.message.content

print('chat() ready')

chat() ready


## 3. A basic call — no system prompt, on purpose

The simplest possible use: a single user message, **nothing else**. No
`system=` argument is passed at all — watch the mock note confirm this
explicitly. This is what "just asking a question" looks like with zero
configuration.

In [24]:
reply = chat([{"role": "user", "content": "What's a good rule of thumb for saving money each month?"}], verbose=True)
print(reply)

[finish_reason: length]
[TRUNCATED -- hit the 1000-token limit before finishing. Raise max_tokens.]
The most popular and practical rule of thumb for saving money is the **50/30/20 Rule**. 

Popularized by Senator Elizabeth Warren in her book *All Your Worth*, this rule suggests dividing your **after-tax (take-home) income** into three clear categories:

---

### The 50/30/


**Notice there was no persona, no instruction, nothing shaping the answer** —
just whatever the model's own default "helpful assistant" behavior happens to
be. That's a completely valid way to call an LLM. The system prompt is
optional, not required — but it's also the single biggest lever you have
when you *do* want to shape the answer, which is exactly what the next
section shows.

## 4. Now, the same question — WITH a system prompt

Same user question as section 3, unchanged. The only thing that's different
this time: a `system=` argument is passed. Watch how much the *personality
and content* of the answer shifts, purely from that one addition.

In [25]:
question = [{"role": "user", "content": "Should I invest my savings in the stock market?"}]

personas = {
    "Cautious financial advisor": "You are a cautious, risk-averse financial advisor. Always mention downside risk first.",
    "Enthusiastic startup founder": "You are an enthusiastic startup founder who thinks everyone should take more risks. Be energetic and brief.",
    "Terse, no-nonsense analyst": "You are a terse financial analyst. Answer in one blunt sentence, no pleasantries.",
}

for name, system_prompt in personas.items():
    print(f'--- {name} ---')
    print(chat(question, system=system_prompt))
    print()

--- Cautious financial advisor ---
Before considering the potential gains of the stock market, **you must first understand the significant downside risks involved.**

Investing in the stock market carries the inherent risk of capital loss. Unlike a FDIC-insured savings account, there are no guarantees in the stock market, and you can lose a substantial portion—or even all—of your principal investment. 

Here are the primary risks you must be prepared for:

1. **Market Volatility and Loss of Capital:** Stock prices fluctuate daily based on economic shifts, corporate performance, and global events. During market downturns or recessions, portfolios can drop by 20%, 30%, or even 50% in a short period. If you need your money during a crash, you may be forced to sell at a severe loss.
2. **Illiquidity in Downturns:** While stocks are technically liquid (you can sell them on any trading day), selling during a market crash locks in real losses. Therefore, money invested in the stock market sho

**Same question, three completely different answers** — and compare all
three to section 3's answer, which had no system prompt at all. Four
variations of the same question, four different outcomes, and the *only*
thing that changed each time was whether (and how) a system prompt was set.

## 5. "Memory" is an illusion you build yourself

The model does not remember anything between calls. Every request is
completely stateless. "Conversation" only works because *you* resend the
entire history every time, with the newest message tacked on the end.

In [26]:
conversation = []

def say(user_text, system=None):
    conversation.append({"role": "user", "content": user_text})
    reply = chat(conversation, system=system)
    conversation.append({"role": "assistant", "content": reply})
    print(f'YOU:  {user_text}')
    print(f'BOT:  {reply}')
    print()
    return reply

SYSTEM = "You are a friendly assistant helping someone plan a trip. Keep replies short."

say("I want to go somewhere warm in December.", system=SYSTEM)
say("Somewhere in Africa specifically.", system=SYSTEM)
say("What did I say I wanted, again?", system=SYSTEM)

YOU:  I want to go somewhere warm in December.
BOT:  December is a great time for a warm getaway! Here are a few top spots to consider:

* **Cancun, Mexico:** Sunny beaches, easy flights, and great resorts.
* **Costa Rica:** Warm weather, beautiful beaches, and lush rainforests.
* **Oahu, Hawaii:** Perfect tropical weather, sightseeing, and surfing.
* **Phuket, Thailand:** Ideal dry-season weather, stunning islands, and rich culture.

What kind of trip are you aiming for—relaxing on a beach, adventure, or exploring a new culture?



RateLimitError: Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3.6-flash\nPlease retry in 5.728998939s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-3.6-flash', 'location': 'global'}, 

In [27]:
import json
print(json.dumps(conversation, indent=2))

[
  {
    "role": "user",
    "content": "I want to go somewhere warm in December."
  },
  {
    "role": "assistant",
    "content": "December is a great time for a warm getaway! Here are a few top spots to consider:\n\n* **Cancun, Mexico:** Sunny beaches, easy flights, and great resorts.\n* **Costa Rica:** Warm weather, beautiful beaches, and lush rainforests.\n* **Oahu, Hawaii:** Perfect tropical weather, sightseeing, and surfing.\n* **Phuket, Thailand:** Ideal dry-season weather, stunning islands, and rich culture.\n\nWhat kind of trip are you aiming for\u2014relaxing on a beach, adventure, or exploring a new culture?"
  },
  {
    "role": "user",
    "content": "Somewhere in Africa specifically."
  }
]


**That printed list is the whole trick.** No hidden state on the server —
just a growing list of messages resent in full, every time.

**Try it yourself:** comment out the line that appends the assistant's reply,
ask a follow-up that depends on earlier context, and watch the model lose the
thread completely — because as far as it knows, this is the first thing
anyone has ever said to it.

In [29]:
from pathlib import Path
import json
import numpy as np

# ============================================================
# SAVE MODEL RESULTS FOR DASHBOARD
# ============================================================

BASE_DIR = Path.cwd()

# If notebook is inside the project, find the project root
while BASE_DIR.name != "2026-orion" and BASE_DIR.parent != BASE_DIR:
    BASE_DIR = BASE_DIR.parent

RESULTS_DIR = BASE_DIR / "dashboard" / "data"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_PATH = RESULTS_DIR / "model_results.json"


def to_list(value):
    if value is None:
        return []

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, (list, tuple)):
        return [
            to_list(x)
            for x in value
        ]

    if isinstance(value, (np.integer, np.floating)):
        return value.item()

    return value


results = {
    "status": "ok",

    "model": {
        "name": "LSTM",
        "sequence_length": 10,
        "top_k": 4,
        "epochs": 150
    },

    "loss": {
        "train": to_list(train_losses),
        "test": to_list(test_losses)
    },

    "final_values": {
        "lstm": 1303.9096,
        "mlp": 2008.3079,
        "benchmark": 1567.1978
    },

    "predictions": to_list(predictions),
    "actual": to_list(actual)
}


with open(
    RESULTS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        indent=2
    )


print("========================================")
print("MODEL RESULTS SAVED")
print("========================================")
print(f"File: {RESULTS_PATH}")
print(f"LSTM final value: {results['final_values']['lstm']}")
print(f"MLP final value: {results['final_values']['mlp']}")
print(f"Benchmark: {results['final_values']['benchmark']}")
print(f"Training loss points: {len(results['loss']['train'])}")
print(f"Testing loss points: {len(results['loss']['test'])}")
print("========================================")

NameError: name 'train_losses' is not defined

In [34]:
# Find variables containing arrays/lists/numbers
# and show their names + shapes/sizes

import numpy as np

print("Possible result variables:")
print("=" * 60)

for name, value in sorted(globals().items()):

    if name.startswith("_"):
        continue

    try:
        if isinstance(value, np.ndarray):
            print(
                f"{name:30} "
                f"numpy array shape={value.shape}"
            )

        elif isinstance(value, (list, tuple)):
            print(
                f"{name:30} "
                f"{type(value).__name__} length={len(value)}"
            )

    except Exception:
        pass

Possible result variables:
In                             list length=35
conversation                   list length=3
question                       list length=1


In [32]:
print("Number of variables:", len(globals()))

print("\nVariables:")
for name in sorted(globals()):
    if not name.startswith("_"):
        print(name)

Number of variables: 82

Variables:
AI_MODEL
BASE_DIR
CONFIG
In
OPENAI_API_KEY
OpenAI
Out
PROVIDER
Path
RESULTS_DIR
RESULTS_PATH
SYSTEM
ai_client
chat
conversation
exit
get_ipython
json
load_dotenv
name
np
open
os
personas
question
quit
reply
say
system_prompt
to_list
